# ⚖️ BatchTopK vs TopK - SAE architecture comparison

This notebook replicates the key experiment from **Bussmann, Nabeshima, Conmy & Nanda, *BatchTopK Sparse Autoencoders* ([arxiv:2412.06410](https://arxiv.org/abs/2412.06410))**. Standard TopK SAEs (Gao et al. 2024) force **exactly `k` active features on every token** - a rigid budget that wastes capacity on easy tokens and under-serves hard ones. BatchTopK instead picks the top `k × B` activations across the *entire batch*, letting the average stay at `k` while individual tokens spend more or fewer features as needed.

We train both variants on the **same activation stream** from Gemma-2-2B layer 15, then compare the Pareto frontier of *variance explained* vs *L0* vs *dead-feature fraction*. Running on a free Colab T4 with a 20M-token budget, the whole comparison finishes in ~45 min and produces two HF-uploaded SAEs plus a side-by-side metric report.

In [ ]:
!pip install -q -U transformers accelerate datasets safetensors einops huggingface_hub matplotlib
!nvidia-smi | head -10

## Configuration

A smaller SAE (`N_FEATURES=8192`, ~4× expansion) and a shorter 20M-token budget keep the run Colab-T4-friendly while still giving a clean Pareto signal between TopK and BatchTopK. The only thing you **must** edit is `HF_USERNAME`. Flip `PUSH_TO_HF=False` to save locally only.

In [ ]:
import torch

MODEL_ID       = 'google/gemma-2-2b'
LAYER          = 15
D_MODEL        = 2304
N_FEATURES     = 8192                # 4x expansion - fast comparison run
K              = 64                  # average L0 target for both variants
TOKEN_BUDGET   = 20_000_000          # ~20-25 min per SAE, ~45 min total
SEQ_LEN        = 512
FWD_BATCH      = 4
BATCH_SIZE     = 2048                # SAE training batch (tokens per step)
LR_PEAK        = 2e-4
LR_FLOOR       = 6e-5
WARMUP_STEPS   = 500
DEAD_TOKENS    = 2_000_000           # stale window for dead-feature counting
LOG_EVERY      = 100                 # log var_exp / L0 / dead every N steps
EVAL_TOKENS    = 500_000             # fresh tokens for final comparison

HF_USERNAME    = 'YOUR_HF_USERNAME'  # <-- USER EDITS THIS
HF_REPO_TOPK   = f'{HF_USERNAME}/gemma2-2b-sae-topk'
HF_REPO_BATCH  = f'{HF_USERNAME}/gemma2-2b-sae-batchtopk'
PUSH_TO_HF     = True                # set False to skip HF upload
LOCAL_OUT      = '/content/sae_compare'

DEVICE = 'cuda'
print(f'Comparing TopK vs BatchTopK on {MODEL_ID} layer {LAYER}')
print(f'd_sae={N_FEATURES}, k={K}, budget={TOKEN_BUDGET:,} tokens per SAE')
print(f'Reference: Bussmann et al. 2024, arxiv:2412.06410')

## Load model + stream activations

Gemma-2-2B in bf16 with SDPA attention fits the T4's 16 GB with room for activation capture. We mount Google Drive (for optional local save), auth to HF, attach a forward hook on layer 15, and expose `activation_stream(...)` that yields `(B·T, D_MODEL)` float32 activation tensors packed from streaming FineWeb-Edu.

In [ ]:
import os
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# Optional Colab auth - safe to skip if running elsewhere
try:
    from google.colab import userdata
    from huggingface_hub import login
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        login(HF_TOKEN)
        print('HF auth OK')
except Exception as e:
    print(f'Skipping Colab auth ({e}) - set HF_TOKEN env var if pushing to HF.')

Path(LOCAL_OUT).mkdir(parents=True, exist_ok=True)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,            # NOTE: `dtype=`, not `torch_dtype=`
    device_map='cuda',
    attn_implementation='sdpa',
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

_act_cache = {}
def _hook(module, inp, out):
    _act_cache['x'] = out[0] if isinstance(out, tuple) else out
hook_handle = model.model.layers[LAYER].register_forward_hook(_hook)

corpus = load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT',
                      split='train', streaming=True)

def token_packer(stream, seq_len=SEQ_LEN, batch=FWD_BATCH):
    buf = []
    for doc in stream:
        ids = tok.encode(doc['text'], add_special_tokens=False)
        buf.extend(ids + [tok.eos_token_id])
        while len(buf) >= batch * seq_len:
            chunk = buf[:batch * seq_len]
            buf = buf[batch * seq_len:]
            yield torch.tensor(chunk, dtype=torch.long).view(batch, seq_len)

def activation_stream(token_iter):
    for ids in token_iter:
        ids = ids.to(DEVICE, non_blocking=True)
        with torch.no_grad():
            model(ids, use_cache=False)
        x = _act_cache['x'].reshape(-1, D_MODEL)
        yield x.float()

vram_gb = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded. VRAM used: {vram_gb:.2f} GB / 16 GB')
print('Activation stream primed.')

## Two SAE classes sharing the same encoder/decoder shape

Both variants share identical weights, init, and forward pass - the **only** difference is how the TopK mask is computed. This keeps the comparison fair: same parameter count, same optimizer, same data.

- **TopKSAE**: for each row of `pre` pick the `k` largest - per-token budget = exactly `k`.
- **BatchTopKSAE**: flatten the whole `(B, d_sae)` pre-activation matrix, pick the `k·B` largest values globally - per-token L0 varies, batch-average stays at `k`.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def geometric_median(X, n_iter=25, eps=1e-6):
    y = X.mean(0)
    for _ in range(n_iter):
        d = (X - y).norm(dim=1).clamp_min(eps)
        y = (X / d[:, None]).sum(0) / (1.0 / d).sum()
    return y

class _BaseSAE(nn.Module):
    """Shared init, decoder renorm, and dead-feature bookkeeping."""
    def __init__(self, d_in, d_sae, k):
        super().__init__()
        self.d_in, self.d_sae, self.k = d_in, d_sae, k
        W = torch.randn(d_in, d_sae)
        W /= W.norm(dim=0, keepdim=True)
        self.W_dec = nn.Parameter(W.T.contiguous())       # (d_sae, d_in)
        self.W_enc = nn.Parameter(W.contiguous())         # (d_in, d_sae)
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.register_buffer('last_fired', torch.zeros(d_sae, dtype=torch.long))
        self._step = 0

    @torch.no_grad()
    def init_b_dec(self, X):
        self.b_dec.data = geometric_median(X).to(self.b_dec)

    def encode_pre(self, x):
        return (x - self.b_dec) @ self.W_enc + self.b_enc

    def _record_firing(self, z):
        fired = (z != 0).any(dim=0)
        self.last_fired[fired] = self._step

    @torch.no_grad()
    def renorm_decoder(self):
        self.W_dec.data /= self.W_dec.data.norm(dim=1, keepdim=True).clamp_min(1e-8)


class TopKSAE(_BaseSAE):
    """Standard per-sample TopK (Gao et al. 2024). Exactly k active per token."""
    def encode(self, x):
        pre = self.encode_pre(x)                           # (B, d_sae)
        vals, idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, F.relu(vals))
        return z

    def forward(self, x):
        z = self.encode(x)
        x_hat = z @ self.W_dec + self.b_dec
        recon = (x - x_hat).pow(2).mean()
        self._record_firing(z)
        return x_hat, recon, z

print('TopKSAE defined.')

In [ ]:
class BatchTopKSAE(_BaseSAE):
    """BatchTopK (Bussmann et al. 2024, arxiv:2412.06410).

    Pick the top `k * B` activations across the FLATTENED (B, d_sae) pre matrix.
    Per-token L0 varies; batch-average stays at k.
    """
    def encode(self, x):
        pre = self.encode_pre(x)                           # (B, d_sae)
        B = x.shape[0]
        flat = pre.flatten()                               # (B * d_sae,)
        total_k = self.k * B
        top_v, top_i = flat.topk(total_k)
        z_flat = torch.zeros_like(flat)
        z_flat.scatter_(0, top_i, F.relu(top_v))
        z = z_flat.reshape(pre.shape)
        return z

    def forward(self, x):
        z = self.encode(x)
        x_hat = z @ self.W_dec + self.b_dec
        recon = (x - x_hat).pow(2).mean()
        self._record_firing(z)
        return x_hat, recon, z

print('BatchTopKSAE defined.')

## Train both SAEs in parallel on the same activation stream

The critical fairness constraint: both SAEs see the **same batches in the same order**. We instantiate them, give each its own Adam, and step both on every batch. Every `LOG_EVERY` steps we record `var_exp`, average L0, and dead-feature fraction for each. This lets us plot the full training trajectory, not just the final numbers.

In [ ]:
import math, time, json
from tqdm.auto import tqdm

sae_topk  = TopKSAE(D_MODEL, N_FEATURES, K).to(DEVICE)
sae_batch = BatchTopKSAE(D_MODEL, N_FEATURES, K).to(DEVICE)
opt_topk  = torch.optim.Adam(sae_topk.parameters(),  lr=LR_PEAK, betas=(0.9, 0.999))
opt_batch = torch.optim.Adam(sae_batch.parameters(), lr=LR_PEAK, betas=(0.9, 0.999))

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR_PEAK * step / max(1, WARMUP_STEPS)
    total = TOKEN_BUDGET // BATCH_SIZE
    p = (step - WARMUP_STEPS) / max(1, total - WARMUP_STEPS)
    p = min(1.0, max(0.0, p))
    return LR_FLOOR + 0.5 * (LR_PEAK - LR_FLOOR) * (1 + math.cos(math.pi * p))

stale_steps = max(1, DEAD_TOKENS // BATCH_SIZE)
history = {'step': [], 'tokens': [],
           'topk_var': [],  'topk_l0': [],  'topk_dead': [],
           'batch_var': [], 'batch_l0': [], 'batch_dead': []}

buf = torch.empty(0, D_MODEL, device=DEVICE)
act_iter = activation_stream(token_packer(corpus))
tokens_seen, step = 0, 0
pbar = tqdm(total=TOKEN_BUDGET, unit='tok', unit_scale=True)
t0 = time.time()

while tokens_seen < TOKEN_BUDGET:
    while buf.shape[0] < BATCH_SIZE:
        buf = torch.cat([buf, next(act_iter)], dim=0)
    x = buf[:BATCH_SIZE]
    buf = buf[BATCH_SIZE:]

    # One-time b_dec init from first real batch (same data -> same init for both)
    if step == 0:
        sae_topk.init_b_dec(x[:4096])
        sae_batch.init_b_dec(x[:4096])

    lr = lr_at(step)
    for g in opt_topk.param_groups:  g['lr'] = lr
    for g in opt_batch.param_groups: g['lr'] = lr
    sae_topk._step = step
    sae_batch._step = step

    # --- TopK step ---
    _, recon_t, z_t = sae_topk(x)
    opt_topk.zero_grad(set_to_none=True)
    recon_t.backward()
    opt_topk.step()
    sae_topk.renorm_decoder()

    # --- BatchTopK step ---
    _, recon_b, z_b = sae_batch(x)
    opt_batch.zero_grad(set_to_none=True)
    recon_b.backward()
    opt_batch.step()
    sae_batch.renorm_decoder()

    tokens_seen += BATCH_SIZE
    step += 1
    pbar.update(BATCH_SIZE)

    if step % LOG_EVERY == 0:
        with torch.no_grad():
            var_x = x.var()
            var_t = (1 - recon_t / var_x).item()
            var_b = (1 - recon_b / var_x).item()
            l0_t  = (z_t != 0).float().sum(-1).mean().item()
            l0_b  = (z_b != 0).float().sum(-1).mean().item()
            dead_t = ((step - sae_topk.last_fired)  > stale_steps).float().mean().item()
            dead_b = ((step - sae_batch.last_fired) > stale_steps).float().mean().item()
        history['step'].append(step); history['tokens'].append(tokens_seen)
        history['topk_var'].append(var_t);   history['batch_var'].append(var_b)
        history['topk_l0'].append(l0_t);     history['batch_l0'].append(l0_b)
        history['topk_dead'].append(dead_t); history['batch_dead'].append(dead_b)
        pbar.set_postfix(topk=f'{var_t:.3f}/L0={l0_t:.1f}/d={dead_t:.2f}',
                         batch=f'{var_b:.3f}/L0={l0_b:.1f}/d={dead_b:.2f}')

pbar.close()
print(f'Training done in {(time.time()-t0)/60:.1f} min over {tokens_seen:,} tokens.')

## Evaluation + comparison on fresh tokens

Training-time metrics can be misleading (they see each batch only once). We now draw `EVAL_TOKENS` **fresh** tokens from the same stream and compute held-out `var_exp`, average L0, L0 distribution, and dead-feature fraction for each SAE on identical activations.

In [ ]:
sae_topk.eval();  sae_batch.eval()

val_buf = torch.empty(0, D_MODEL, device=DEVICE)
val_iter = activation_stream(token_packer(corpus))
with tqdm(total=EVAL_TOKENS, desc='collecting eval acts', unit='tok', unit_scale=True) as pb:
    while val_buf.shape[0] < EVAL_TOKENS:
        new = next(val_iter)
        val_buf = torch.cat([val_buf, new], dim=0)
        pb.update(new.shape[0])
x_val = val_buf[:EVAL_TOKENS]

def evaluate(sae, x_val, chunk=8192):
    """Chunked eval to keep VRAM low on T4."""
    n = x_val.shape[0]
    total_var = x_val.var().item()
    sse = 0.0
    l0_sum = 0.0
    l0_samples = []
    fire_any = torch.zeros(sae.d_sae, dtype=torch.bool, device=x_val.device)
    with torch.no_grad():
        for i in range(0, n, chunk):
            xc = x_val[i:i+chunk]
            x_hat, recon, z = sae(xc)
            sse += recon.item() * xc.numel()
            l0_tok = (z != 0).float().sum(-1)
            l0_sum += l0_tok.sum().item()
            l0_samples.append(l0_tok.cpu())
            fire_any |= (z != 0).any(dim=0)
    mse = sse / (n * sae.d_in)
    var_exp = 1 - mse / total_var
    avg_l0 = l0_sum / n
    dead_frac = 1.0 - fire_any.float().mean().item()
    return {
        'var_exp':   float(var_exp),
        'avg_l0':    float(avg_l0),
        'dead_frac': float(dead_frac),
        'l0_samples': torch.cat(l0_samples).numpy(),
    }

eval_topk  = evaluate(sae_topk,  x_val)
eval_batch = evaluate(sae_batch, x_val)

print(f'{"metric":<14} {"TopK":>12} {"BatchTopK":>12}   delta')
print('-' * 56)
for key in ['var_exp', 'avg_l0', 'dead_frac']:
    dt = eval_batch[key] - eval_topk[key]
    print(f'{key:<14} {eval_topk[key]:>12.4f} {eval_batch[key]:>12.4f}   {dt:+.4f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Plot 1: var_exp trajectory during training ---
ax = axes[0]
ax.plot(history['tokens'], history['topk_var'],  label='TopK',     linewidth=2)
ax.plot(history['tokens'], history['batch_var'], label='BatchTopK', linewidth=2)
ax.set_xlabel('Tokens seen')
ax.set_ylabel('Variance explained')
ax.set_title(f'Training trajectory (k={K}, d_sae={N_FEATURES})')
ax.legend(); ax.grid(alpha=0.3)

# --- Plot 2: held-out L0 distribution ---
ax = axes[1]
bins = np.arange(0, max(eval_topk['l0_samples'].max(), eval_batch['l0_samples'].max()) + 2) - 0.5
ax.hist(eval_topk['l0_samples'],  bins=bins, alpha=0.55, label=f"TopK (avg={eval_topk['avg_l0']:.1f})")
ax.hist(eval_batch['l0_samples'], bins=bins, alpha=0.55, label=f"BatchTopK (avg={eval_batch['avg_l0']:.1f})")
ax.axvline(K, color='k', linestyle='--', alpha=0.5, label=f'target k={K}')
ax.set_xlabel('Per-token L0 (active features)')
ax.set_ylabel('Tokens')
ax.set_title('Held-out L0 distribution')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{LOCAL_OUT}/comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved plot to {LOCAL_OUT}/comparison.png')

## What to look for

The Bussmann et al. result (arxiv:2412.06410) is that **BatchTopK Pareto-dominates standard TopK at the same average L0** - it reaches higher variance explained for the same feature budget, and typically has a lower dead-feature fraction because the cross-batch competition lets rarely-useful-but-occasionally-critical features survive.

You should see:
- **var_exp**: BatchTopK higher (a few percentage points at this budget; the gap widens with longer training).
- **avg_l0**: both hovering near `K=64` on the held-out set - BatchTopK by design, TopK by construction.
- **L0 histogram**: TopK is a delta at `K`; BatchTopK is a spread *centred* on `K`, with some tokens using far fewer features and some a lot more. This is the mechanism.
- **dead_frac**: BatchTopK lower, especially with a tight budget like this.

If the gap is tiny, extend `TOKEN_BUDGET` to 50M+ - the advantage compounds with scale.

In [ ]:
from safetensors.torch import save_file
from huggingface_hub import HfApi, create_repo

def save_sae(sae, arch, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    state = {
        'W_enc': sae.W_enc.detach().cpu().contiguous(),
        'W_dec': sae.W_dec.detach().cpu().contiguous(),
        'b_enc': sae.b_enc.detach().cpu().contiguous(),
        'b_dec': sae.b_dec.detach().cpu().contiguous(),
    }
    save_file(state, str(out_dir / 'sae.safetensors'))
    cfg = {
        'architecture': arch,
        'd_in': D_MODEL, 'd_sae': N_FEATURES, 'k': K,
        'hook_name': f'blocks.{LAYER}.hook_resid_post',
        'model_name': MODEL_ID,
        'tokens_trained': tokens_seen,
        'reference': 'Bussmann et al. 2024, arxiv:2412.06410' if arch == 'batchtopk'
                     else 'Gao et al. 2024, arxiv:2406.04093',
    }
    (out_dir / 'cfg.json').write_text(json.dumps(cfg, indent=2))
    return out_dir

topk_dir  = save_sae(sae_topk,  'topk',      f'{LOCAL_OUT}/topk')
batch_dir = save_sae(sae_batch, 'batchtopk', f'{LOCAL_OUT}/batchtopk')

# Side-by-side JSON report (strip numpy arrays so it serialises cleanly)
def _clean(d):  return {k: v for k, v in d.items() if k != 'l0_samples'}
report = {
    'model': MODEL_ID, 'layer': LAYER,
    'd_sae': N_FEATURES, 'k': K,
    'tokens_trained': tokens_seen, 'eval_tokens': EVAL_TOKENS,
    'topk':      _clean(eval_topk),
    'batchtopk': _clean(eval_batch),
    'delta_var_exp':   eval_batch['var_exp']   - eval_topk['var_exp'],
    'delta_avg_l0':    eval_batch['avg_l0']    - eval_topk['avg_l0'],
    'delta_dead_frac': eval_batch['dead_frac'] - eval_topk['dead_frac'],
    'history': history,
    'reference': 'Bussmann, Nabeshima, Conmy, Nanda - BatchTopK SAEs, arxiv:2412.06410',
}
report_path = Path(LOCAL_OUT) / 'comparison_report.json'
report_path.write_text(json.dumps(report, indent=2))

print('\n=== Summary ===')
print(f'{"":<18} {"var_exp":>10} {"avg_L0":>10} {"dead_frac":>11}')
print(f'{"TopK":<18} {eval_topk["var_exp"]:>10.4f} {eval_topk["avg_l0"]:>10.2f} {eval_topk["dead_frac"]:>11.4f}')
print(f'{"BatchTopK":<18} {eval_batch["var_exp"]:>10.4f} {eval_batch["avg_l0"]:>10.2f} {eval_batch["dead_frac"]:>11.4f}')
print(f'{"delta (batch-topk)":<18} {report["delta_var_exp"]:>+10.4f} {report["delta_avg_l0"]:>+10.2f} {report["delta_dead_frac"]:>+11.4f}')
print(f'\nReport saved: {report_path}')

# --- HF upload ---
if PUSH_TO_HF:
    api = HfApi()
    for repo_id, src_dir in [(HF_REPO_TOPK, topk_dir), (HF_REPO_BATCH, batch_dir)]:
        try:
            create_repo(repo_id, exist_ok=True, private=False)
            for f in ['sae.safetensors', 'cfg.json']:
                api.upload_file(path_or_fileobj=str(src_dir / f), path_in_repo=f,
                                repo_id=repo_id, repo_type='model')
            api.upload_file(path_or_fileobj=str(report_path),
                            path_in_repo='comparison_report.json',
                            repo_id=repo_id, repo_type='model')
            print(f'Uploaded -> https://huggingface.co/{repo_id}')
        except Exception as e:
            print(f'HF push failed for {repo_id}: {e}')
            print(f'Local copy kept at {src_dir}')
else:
    print(f'PUSH_TO_HF=False - SAEs kept locally at {LOCAL_OUT}/')

hook_handle.remove()